#Initialize

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

#Read bronze table

In [0]:
df = spark.table("erp.bronze.erp_loc_a101")

#Silver Transformations

In [0]:
#Trimming

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
     
#Customer ID Cleanup

df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))
     
#Country Normalization

df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)
     
#Renaming Columns

RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
     
#Sanity checks of dataframe

df.limit(10).display()
     

#writing silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("erp.silver.erp_customer_location")